In [1]:
import h5py
import numpy as np
import scipy.io
import scipy.signal
import h5py

f = h5py.File('../data/Part_1.mat', 'r')
list(f.keys())

['#refs#', 'Part_1']

In [ ]:
class PPGRecordCleaner:
    """Cleans and normalizes one patient record (PPG + ABP) from the UCI dataset."""

    def __init__(self, record: np.ndarray, fs: int = 125):
        # Each record is a table of 61,000 rows x 3 columns — 61,000 because
        # the sensors record 125 times per second, and this recording runs
        # about 8 minutes. The 3 columns are PPG, ABP, and ECG
        # (we're ignoring ECG per the project brief — don't need it).
        self.fs = fs
        self.ppg = record[:, 0]
        self.abp = record[:, 1]

        # populated as methods run
        self.abp_bad = None
        self.ppg_bad_full = None
        self.ppg_clean = None
        self.abp_clean = None
        self.ppg_norm = None

    def check_missing(self) -> dict:
        # Zero NaNs doesn't mean the data is clean — it just means there's
        # no explicit "this is missing" (NaN) flag. Real problems in sensor
        # data usually show up as garbage values, not blank ones.
        return {
            "ppg_nan": int(np.isnan(self.ppg).sum()),
            "abp_nan": int(np.isnan(self.abp).sum()),
        }

    def detect_outliers(self, abp_bounds=(30, 300), k=3):
        # Two different checks, because PPG and ABP need different rules:
        # - ABP has known physiological limits (blood pressure can't
        #   reasonably be below 30 or above 300 mmHg), so we check against
        #   that fixed range.
        # - PPG has no fixed "normal range" — it's a relative sensor
        #   reading — so we use a statistical rule instead (IQR: flag
        #   anything unusually far from the typical spread of values).
        lower, upper = abp_bounds
        self.abp_bad = (self.abp < lower) | (self.abp > upper) | np.isnan(self.abp)

        valid = self.ppg[~np.isnan(self.ppg)]
        q1, q3 = np.percentile(valid, [25, 75])
        iqr = q3 - q1
        ppg_lower, ppg_upper = q1 - k * iqr, q3 + k * iqr
        ppg_bad = (self.ppg < ppg_lower) | (self.ppg > ppg_upper) | np.isnan(self.ppg)

        # Explicit clipping check, on top of IQR: the PPG minimum landing
        # exactly on 0.0 (and max landing on a suspiciously exact ceiling)
        # is often a sign of clipping — the sensor hit its hard limit and
        # got "pinned" there, rather than genuinely reading that value.
        # IQR alone misses this, since a clipped value can still fall
        # within the statistically "normal" range.
        ppg_clip = (self.ppg == self.ppg.min()) | (self.ppg == self.ppg.max())
        self.ppg_bad_full = ppg_bad | ppg_clip
        return self.abp_bad, self.ppg_bad_full

    def summary(self) -> dict:
        """QC stats for documentation — useful for the Sept milestone writeup."""
        return {
            "abp_flagged_pct": round(self.abp_bad.mean() * 100, 4) if self.abp_bad is not None else None,
            "ppg_flagged_pct": round(self.ppg_bad_full.mean() * 100, 4) if self.ppg_bad_full is not None else None,
        }

In [ ]:
cleaner = PPGRecordCleaner(record_np)

missing = cleaner.check_missing()
print("Missing values (NaN count):")
print("  PPG:", missing["ppg_nan"])
print("  ABP:", missing["abp_nan"])

abp_bad, ppg_bad_full = cleaner.detect_outliers()
print(f"\nABP flagged: {abp_bad.sum()} / {len(abp_bad)} ({abp_bad.mean()*100:.3f}%)")
print(f"PPG flagged (IQR + clipping): {ppg_bad_full.sum()} / {len(ppg_bad_full)} ({ppg_bad_full.mean()*100:.3f}%)")

Missing values (NaN count):
  PPG: 0
  ABP: 0

ABP flagged: 0 / 61000 (0.000%)
PPG flagged (IQR + clipping): 30 / 61000 (0.049%)


The file contains 3000 separate records — think of each one as roughly one patient's recording session. Each record is a table of 61,000 rows × 3 columns — 61,000 because the sensors record 125 times per second, and this recording runs about 8 minutes. The 3 columns are the three signals: PPG, ABP, and ECG (we're ignoring ECG per the project brief — don't need it).

In [27]:
# --- Investigating the clipping clusters found in this record's PPG signal ---
# We noticed PPG's minimum value was exactly 0.0 — a suspiciously round number.
# That's often a sign of clipping: the sensor hit its hard limit and got
# "pinned" there, rather than genuinely reading a true value of zero.
# We counted 15 samples at that floor and 15 at the ceiling — a symmetric
# pattern, consistent with a brief moment where the sensor lost proper
# contact (like the finger shifting).

zero_idx = np.where(ppg == 0.0)[0]
max_idx = np.where(ppg == ppg.max())[0]

print("Indices at 0.0:", zero_idx)
print("Indices at max:", max_idx)
print("Span of 0.0 cluster (samples):", zero_idx.max() - zero_idx.min())
print("Span of max cluster (samples):", max_idx.max() - max_idx.min())

# Result:
#   Indices at 0.0: [23657 ... 23671] — 15 samples, span 14 (fully contiguous, one clean block)
#   Indices at max: [20948-20953, 23503-23511] — actually TWO separate groups,
#     not one cluster, despite the large combined span (2563) suggesting otherwise
#
# We then found where in time these sat, and discovered two separate incidents:
#   - A short, isolated 6-sample blip (20948-20953), ~20 sec away from anything else
#   - A more serious event: max-clip (23503-23511) followed ~1.17 sec later by
#     min-clip (23657-23671) — likely one continuous motion artifact (sensor
#     briefly losing/regaining contact), since 1.17 sec is within a single
#     heartbeat cycle


# --- Confirming which 625-sample windows these fall into ---
window_size = 625
print("Window for first max cluster (20948):", 20948 // window_size)
print("Window for second max cluster (23503):", 23503 // window_size)
print("Window for 0.0 cluster (23657):", 23657 // window_size)

# Result:
#   Window 33 — isolated 6-sample max-clip blip (contained, minor)
#   Window 37 — compound event: 9-sample max-clip + 15-sample min-clip,
#     24 of our 30 flagged samples live here — the more severe artifact
#
# This confirms the two max groups belong to genuinely different windows,
# and that the second max group + the 0.0 group are likely connected
# (same window, ~1 sec apart) rather than coincidental.

Indices at 0.0: [23657 23658 23659 23660 23661 23662 23663 23664 23665 23666 23667 23668
 23669 23670 23671]
Indices at max: [20948 20949 20950 20951 20952 20953 23503 23504 23505 23506 23507 23508
 23509 23510 23511]
Span of 0.0 cluster (samples): 14
Span of max cluster (samples): 2563
Window for first max cluster (20948): 33
Window for second max cluster (23503): 37
Window for 0.0 cluster (23657): 37
